# From notebook to application, Git and continuous integration

After this block you can turn a notebook analysis into a module and a structured project, and explain when a notebook is the right tool and when an application is. You can record your work in Git, work on branches, merge them and resolve a merge conflict, propose and review changes as pull requests on GitHub, and set up a continuous-integration workflow that runs the linter and the tests on every change.

*Session 2, block 2 of 2. Course material, CC BY 4.0; sections adapted from third-party material say so in an attribution line.*

## Overview

| Section | What it covers |
|---|---|
| From notebook to application: when to use which | hidden state, explicit inputs, a script runs in a fresh process, when a notebook and when a module |
| Scripts, modules, packages and project structure | scripts and modules, `__name__ == "__main__"`, how Python finds modules, `argparse`, packages, the workspace layout, `pyproject.toml`, packaging and installing, the `src` layout, the package as a command-line program |
| Git: commits, branches, merging and merge conflicts | repositories, commits and the staging area, history and hashes, `.gitignore`, branches, fast-forward and merge commits, merge conflicts, comparing branches, undoing, remotes and a rejected push |
| GitHub: pull requests and code review | pull requests, descriptions and issues, pull-request states, fork and pull, code review and its etiquette |
| Continuous integration with GitHub Actions | "it works on my machine", workflows, test matrices, linting and formatting with ruff, branch protection, notebooks and secrets |
| Worked example | ruff, pytest and the command-line program on a fresh copy of the workspace |
| Check your understanding, Exercises, Debrief | five questions, six exercises in pairs, what to compare afterwards |

The first code cell moves Python to the repository root, so that the paths of the case-study data and of the workspace work on every computer.

In [1]:
# Run the code from the repository root, where the case-study paths start.
import os
from pathlib import Path

os.chdir(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").is_dir()))

Many cells in this block run terminal commands (`python`, `git`, `ruff`, `pytest`). `subprocess.run` starts a program as if you had typed it in a terminal; the helper `run` prints what the terminal would show. Every file these cells create lives in a temporary folder made by `tempfile.mkdtemp()`, never in the repository.

In [2]:
import shutil
import subprocess
import sys
import tempfile

WORKSPACE = Path("sessions/02-software-engineering/workspace")


def run(*cmd, cwd=None, env=None):
    """Run a command as in a terminal and print its output."""
    plain = {k: v for k, v in os.environ.items() if k not in ("FORCE_COLOR", "CLICOLOR_FORCE")}
    result = subprocess.run([str(c) for c in cmd], cwd=cwd, capture_output=True, text=True,
                            env={**plain, "PYTHONDONTWRITEBYTECODE": "1", "NO_COLOR": "1", **(env or {})})
    output = (result.stdout + result.stderr).rstrip()
    if output:
        print(output)

## From notebook to application: when to use which

A notebook is like a lab notebook: good for exploring data, trying ideas and explaining a result with text, code and charts side by side. An application, a module, a command-line program or a service, is like an appliance: other people and programs rely on it, call it with their own inputs and expect the same behaviour every time.

### Hidden state

The difference is **state**: the values a program remembers between steps. A notebook's kernel remembers every name created by any cell that has run, in the order you ran them, not the order on the page. A result can therefore depend on a cell you have since edited or deleted. A script starts a fresh Python process each time and runs from top to bottom. Changing the file extension from `.ipynb` to `.py` is not enough, though: reproducible code needs **explicit inputs** (parameters instead of global variables), a **documented environment** (`pyproject.toml` and `uv.lock`) and **checks** (tests).

Next, hidden state on real data. `n_short_stays` reads the global variable `max_nights` instead of receiving it as a parameter.

In [3]:
import pandas as pd

listings = pd.read_parquet("case-study/data/airbnb/listings.parquet")
max_nights = 28


def n_short_stays(df):
    return int((df["minimum_nights"] < max_nights).sum())     # reads a global variable


print(n_short_stays(listings))

6632


**Predict before running.** Much later in the notebook, someone reuses the name `max_nights` for another question. What does the same call return now?

In [4]:
max_nights = 3            # a later cell, written for a different question
print(n_short_stays(listings))

4494


The function's answer changed although the function did not. The fix is an explicit parameter with a documented default: now the call says what it depends on, and it can be tested.

In [5]:
def n_short_stays(df: pd.DataFrame, max_nights: int = 28) -> int:
    """Number of listings bookable for fewer than ``max_nights`` nights."""
    return int((df["minimum_nights"] < max_nights).sum())


print(n_short_stays(listings), n_short_stays(listings, max_nights=3))

6632 4494


### A script runs in a fresh process

A script, by contrast, runs in a fresh process. The workspace contains the Session 1 analysis written as notebook-style cells in a plain file, [`notebooks/explore_listings.py`](../workspace/notebooks/explore_listings.py) (`# %%` marks a cell for VS Code). Running it as a program shows that it works from top to bottom with nothing remembered from earlier.

In [6]:
run(sys.executable, WORKSPACE / "notebooks" / "explore_listings.py")

(10221, 42) 6660 hosts
district
Mitte                       2268
Friedrichshain-Kreuzberg    2119
Pankow                      1562
Charlottenburg-Wilm.        1146
Neukölln                    1046
Name: count, dtype: int64
room_type
Entire home/apt    185.0
Hotel room         182.0
Private room        98.0
Shared room         54.0
Name: price, dtype: float64


### When to use which

| Use a notebook to | Move code into a module or package when |
|---|---|
| explore a new dataset and try ideas | the same lines are copied into a second notebook |
| explain a result with text, tables and charts | a teammate, a scheduled job or a service must call the code |
| teach and present | a rule (such as "short stay = under 28 nights") must be the same everywhere |
| record one analysis for a report | the code needs tests, or its inputs change from run to run |

A notebook stays valuable after the logic has moved into a module: it states the question, calls tested functions, draws the chart and interprets the result. The question is not whether a file ends in `.ipynb`, but whether the result can be reproduced and maintained.

## Scripts, modules, packages and project structure

Analysis code becomes reusable in steps: a script, a module, a package, and a project around it. Each step is defined in its own subsection below.

*Adapted from [Python outside the notebook](https://github.com/alan-turing-institute/rse-course/blob/main/module06_software_projects/06_03_non_notebook_python.ipynb) (The Alan Turing Institute, *Research Software Engineering with Python*, CC BY 3.0); local copy: [extras/14-python-outside-the-notebook.ipynb](extras/14-python-outside-the-notebook.ipynb).*

### Scripts and modules

- A **script** is a `.py` file run as a program: `python report.py`.
- A **module** is a `.py` file that other code imports: `import prices`. The same file can be both.

Importing a module runs its top-level statements once. A reusable module should therefore define functions and classes at the top level and put work that should happen only when it is *run* into a function `main()`, called under `if __name__ == "__main__":`. Python sets the special variable `__name__` to `"__main__"` in the file that was started, and to the module's name in a file that was imported.

Next we write a module `prices.py` into a temporary folder. `%%writefile path` saves the cell as a file instead of running it; `{demo}` inserts the folder. The first line prints `__name__` so that we can see who is running the file.

In [7]:
demo = Path(tempfile.mkdtemp(prefix="modules-demo-"))

In [8]:
%%writefile {demo}/prices.py
print("executing prices.py, __name__ is", __name__)


def stay_cost(nightly_price, nights, cleaning_fee=25):
    return nightly_price * nights + cleaning_fee


def main():
    print("a 3-night stay at 80 EUR costs", stay_cost(80, 3))


if __name__ == "__main__":
    main()

Writing /var/folders/_2/pj2r648n4696ddzb0qltgpdr0000gn/T/modules-demo-a9xav0ak/prices.py


**Predict before running.** We run the file as a program, then import it from a second program, `report.py`. Which lines does each print?

In [9]:
run(sys.executable, "prices.py", cwd=demo)

executing prices.py, __name__ is __main__
a 3-night stay at 80 EUR costs 265


In [10]:
%%writefile {demo}/report.py
from prices import stay_cost

print("report: a week at 75 EUR costs", stay_cost(75, 7))

Writing /var/folders/_2/pj2r648n4696ddzb0qltgpdr0000gn/T/modules-demo-a9xav0ak/report.py


In [11]:
run(sys.executable, "report.py", cwd=demo)

executing prices.py, __name__ is prices
report: a week at 75 EUR costs 550


When imported, the whole of `prices.py` is executed, so its first `print` runs, but `__name__` is now `"prices"` and `main()` is skipped. Without the `if` line, every program that imports `prices` would also print the three-night example.

### How Python finds modules

`import prices` searches the folders in `sys.path` in order: first the folder of the script that was started (or the current folder in a notebook), then the installed packages. A local file with the name of a library therefore **shadows** the library. Next, a file called `statistics.py` in our demo folder hides the standard-library module of the same name.

In [12]:
(demo / "statistics.py").write_text("print('this is the local statistics.py, not the standard library')\n")
run(sys.executable, "-c", "import statistics; print(statistics.median([1, 3, 10]))", cwd=demo)
(demo / "statistics.py").unlink()

this is the local statistics.py, not the standard library
Traceback (most recent call last):
  File "<string>", line 1, in <module>
    import statistics; print(statistics.median([1, 3, 10]))
                             ^^^^^^^^^^^^^^^^^
AttributeError: module 'statistics' has no attribute 'median' (consider renaming '/private/var/folders/_2/pj2r648n4696ddzb0qltgpdr0000gn/T/modules-demo-a9xav0ak/statistics.py' since it has the same name as the standard library module named 'statistics' and prevents importing that standard library module)


The import found the local file, and `statistics.median` no longer exists. Never name your own files `pandas.py`, `json.py`, `statistics.py` or after any other library you use.

### Command-line interfaces with `argparse`

`argparse`, in the standard library, turns a function into a program with arguments: positional arguments are required, options such as `--nights` are optional, and `--help` is generated for you. (The package `click` is a popular alternative.) The workspace's `cli.py` uses `argparse` to accept the path of a listings file.

*Adapted from [Python outside the notebook](https://github.com/alan-turing-institute/rse-course/blob/main/module06_software_projects/06_03_non_notebook_python.ipynb) (The Alan Turing Institute, *Research Software Engineering with Python*, CC BY 3.0); local copy: [extras/14-python-outside-the-notebook.ipynb](extras/14-python-outside-the-notebook.ipynb).*

In [13]:
%%writefile {demo}/command.py
from argparse import ArgumentParser

from prices import stay_cost


def process():
    parser = ArgumentParser(description="Cost of a stay in a Berlin listing.")
    parser.add_argument("price", type=float, help="price per night in EUR")         # required
    parser.add_argument("--nights", "-n", type=int, default=1, help="number of nights")
    parser.add_argument("--no-fee", action="store_true", help="leave out the cleaning fee")
    args = parser.parse_args()
    print(stay_cost(args.price, args.nights, 0 if args.no_fee else 25))


if __name__ == "__main__":
    process()

Writing /var/folders/_2/pj2r648n4696ddzb0qltgpdr0000gn/T/modules-demo-a9xav0ak/command.py


In [14]:
run(sys.executable, "command.py", "--help", cwd=demo)

executing prices.py, __name__ is prices
usage: command.py [-h] [--nights NIGHTS] [--no-fee] price

Cost of a stay in a Berlin listing.

positional arguments:
  price                price per night in EUR

options:
  -h, --help           show this help message and exit
  --nights, -n NIGHTS  number of nights
  --no-fee             leave out the cleaning fee


In [15]:
run(sys.executable, "command.py", "80", "--nights", "3", cwd=demo)
run(sys.executable, "command.py", "80", "-n", "3", "--no-fee", cwd=demo)
run(sys.executable, "command.py", "eighty", cwd=demo)

executing prices.py, __name__ is prices
265.0


executing prices.py, __name__ is prices
240.0


executing prices.py, __name__ is prices
usage: command.py [-h] [--nights NIGHTS] [--no-fee] price
command.py: error: argument price: invalid float value: 'eighty'


`argparse` converted the arguments with `type=float` and refused `"eighty"` with a usage message. The `print` at the top of `prices.py` still runs, because `command.py` imports it.

### Packages and the project structure

- A **package** is a folder of modules with one import name, such as `listingtools`; a file `__init__.py` marks the folder as a regular package and runs when the package is imported.
- A **project** is the folder around the package: `pyproject.toml` (name, version, dependencies), `uv.lock` (exact versions), `src/` (the package), `tests/`, a README and a licence.

Next, the structure of the course [workspace](../workspace/README.md), a real project. The helper lists the files and leaves out generated folders such as `.venv`.

In [16]:
def tree(folder, prefix=""):
    skip = {".venv", "__pycache__", ".pytest_cache", ".ruff_cache", ".DS_Store"}
    entries = sorted(p for p in folder.iterdir() if p.name not in skip)
    for i, path in enumerate(entries):
        last = i == len(entries) - 1
        print(prefix + ("└── " if last else "├── ") + path.name)
        if path.is_dir():
            tree(path, prefix + ("    " if last else "│   "))


tree(WORKSPACE)

├── .github
│   └── workflows
│       └── ci.yml
├── .gitignore
├── .python-version
├── README.md
├── notebooks
│   └── explore_listings.py
├── pyproject.toml
├── solutions
│   ├── listings.py
│   ├── records.py
│   └── text_features.py
├── src
│   └── listingtools
│       ├── __init__.py
│       ├── __main__.py
│       ├── cli.py
│       ├── errors.py
│       ├── listings.py
│       ├── records.py
│       └── text_features.py
├── tests
│   ├── conftest.py
│   ├── test_listings.py
│   ├── test_listings_exercises.py
│   ├── test_records.py
│   └── test_text_features.py
└── uv.lock


Each part has one role:

| Part | Role |
|---|---|
| `pyproject.toml` | the project's name, version, Python version, dependencies, development tools and their settings |
| `uv.lock` | the exact version of every package, so that everyone (and CI) installs the same environment |
| `.python-version` | the Python version uv and CI use |
| `src/listingtools/` | the package: code meant to be imported |
| `tests/` | pytest tests; `conftest.py` holds the shared test data |
| `notebooks/` | exploratory code that calls the package |
| `.github/workflows/ci.yml` | the continuous-integration workflow (see the section on continuous integration) |
| `solutions/` | reference solutions of the exercises, for self-checking |

Next we print `pyproject.toml`. `[project]` describes the package and its dependencies; `[project.scripts]` turns the function `main` in `listingtools/cli.py` into a terminal command `listing-summary`; `[dependency-groups]` lists tools only developers need; the `[tool....]` tables configure ruff and pytest.

In [17]:
print((WORKSPACE / "pyproject.toml").read_text())

[project]
name = "listingtools"
version = "0.3.0"
description = "Validated Airbnb listings, a table of listings with summaries and title features (Session 2 workspace)"
readme = "README.md"
requires-python = ">=3.14"
dependencies = [
    "pandas>=3.0",
    "pyarrow>=26.0",
]

[project.scripts]
listing-summary = "listingtools.cli:main"

[build-system]
requires = ["uv_build>=0.9,<0.12"]
build-backend = "uv_build"

[dependency-groups]
dev = [
    "pytest>=9.1",
    "ruff>=0.17",
]

[tool.ruff]
line-length = 100

[tool.ruff.lint]
select = ["E", "F", "I", "B", "UP"]

[tool.pytest.ini_options]
testpaths = ["tests"]



#### Packaging and installing a project

To use a package from anywhere, and to share it, it must be **installed**. The tool that turns the project folder into an installable package is the **build backend**, named in `[build-system]` (the workspace uses `uv_build`; setuptools, hatchling and flit are common alternatives). For new projects, the configuration belongs in `pyproject.toml`; older projects use `setup.py` or `setup.cfg`.

- `[project]` must give at least `name` and `version`; `requires-python` and `dependencies` say what the package needs to run.
- Development tools (pytest, ruff) go into `[dependency-groups]` (or the older `[project.optional-dependencies]`), so that users of the package do not install them.
- `[project.scripts]` creates terminal commands: `listing-summary = "listingtools.cli:main"` means "call `main` in `listingtools/cli.py`".
- `__init__.py` runs on import. The workspace uses it to re-export the main names, so that users write `from listingtools import ListingTable` instead of `from listingtools.listings import ListingTable`. A folder inside the package with its own `__init__.py` is a subpackage (`listingtools.io`, for example).

Installing: in the project folder, `uv sync` creates `.venv`, installs the locked dependencies and installs the project itself in **editable** mode, so changes to the source take effect without reinstalling. With pip, the equivalent is `pip install -e ".[dev]"`. A package in a public Git repository can be installed directly (`uv add git+https://github.com/<owner>/<repo>`), and a package can be published to PyPI so that anyone can `pip install` it. PyPI has almost no review process: be wary of installing unknown packages, and check the spelling of package names.

A project that others use also needs a `README.md` (what it does, how to install and run it), a `LICENSE` file (what others may do with it) and, for research code, a `CITATION.cff` (how to cite it).

*Adapted from [Packaging](https://github.com/alan-turing-institute/rse-course/blob/main/module06_software_projects/06_04_packaging.ipynb) (The Alan Turing Institute, *Research Software Engineering with Python*, CC BY 3.0); local copy: [extras/15-packaging.ipynb](extras/15-packaging.ipynb).* The original uses setuptools and pip; this version uses uv, as the course does.

#### Why a `src` layout

In a **flat layout** the package folder sits next to `pyproject.toml`; in the **`src` layout** it sits inside `src/`. With a flat layout, running the tests from the project folder imports the package from the folder directly (it is on `sys.path`), so the tests may pass even if the *installed* package is broken, for example because a file is missing from it. The `src` layout forces the tests to use the installed package. It is a choice, recommended by the Python Packaging User Guide for packages, not a requirement for a one-file script.

### Importing and running the package

Next we import the package and answer two Session 1 questions with its class `ListingTable`. Inside the workspace, `uv run` makes the package importable; here we add its `src` folder to `sys.path`, the list of folders Python searches for modules. `sys.dont_write_bytecode = True` stops Python from writing cache files into the repository.

In [18]:
sys.dont_write_bytecode = True
sys.path.insert(0, str(WORKSPACE / "src"))
from listingtools import SHORT_STAY_MAX_NIGHTS, ListingTable

table = ListingTable.from_parquet("case-study/data/airbnb/listings.parquet")
print(table, "| short stay means fewer than", SHORT_STAY_MAX_NIGHTS, "nights")
table.price_by_room_type()

ListingTable(10221 listings) | short stay means fewer than 28 nights


,n_listings,median_price
room_type,,
Entire home/apt,3694,185.0
Hotel room,63,182.0
Private room,1483,98.0
Shared room,62,54.0


The 28-night rule now lives in one constant, used by every method; in the notebook-style file it is a literal `< 28` that every copy of the notebook repeats. The same package also runs as a command-line program. `python -m listingtools` runs the package's `__main__.py`, which is what the installed command `listing-summary` does; `PYTHONPATH` tells the new process where the package is.

In [19]:
run(sys.executable, "-m", "listingtools", env={"PYTHONPATH": str(WORKSPACE / "src")})

ListingTable(10221 listings)
{'n_listings': 10221,
 'n_hosts': 6660,
 'listings_per_district': {'Mitte': 2268,
                           'Friedrichshain-Kreuzberg': 2119,
                           'Pankow': 1562,
                           'Charlottenburg-Wilm.': 1146,
                           'Neukölln': 1046},
 'room_type_shares': {'Entire home/apt': 0.692,
                      'Private room': 0.294,
                      'Hotel room': 0.007,
                      'Shared room': 0.007},
 'n_short_stays': 5302,
 'median_short_stay_price': 157.5,
 'median_price_by_room_type': {'Entire home/apt': 185.0,
                               'Hotel room': 182.0,
                               'Private room': 98.0,
                               'Shared room': 54.0}}

Short-stay prices by room type (EUR per night):
                 n_listings  median_price
room_type                                
Entire home/apt        3694         185.0
Hotel room               63         182.0
Private ro

> 📖 **Read more:** [extras/14-python-outside-the-notebook.ipynb](extras/14-python-outside-the-notebook.ipynb) and [extras/15-packaging.ipynb](extras/15-packaging.ipynb) (Turing RSE course): the `greeter` example as script, command-line program and installed package.
>
> 📖 **Read more:** [Session 1, extras/08-python-modules-and-packages.ipynb](../../01-careers-and-python/workbooks/extras/08-python-modules-and-packages.ipynb) (A Whirlwind Tour of Python): the forms of the `import` statement, the standard library and third-party packages.

## Git: commits, branches, merging and merge conflicts

**Version control** records the history of a set of files, so that any earlier state can be restored and compared, and so that several people can change the same files without losing each other's work. It replaces numbered copies such as `analysis_final_v3.ipynb`. **Git** is the version control system; **GitHub** is a web service that hosts Git repositories and adds collaboration (pull requests, reviews, issues, Actions). GitLab and Codeberg offer the same.

Version control gives you, in short: a better kind of backup; a history that answers "when did I introduce this bug?"; any older version can be restored; mistakes can be undone; several versions can be maintained at once; and several people can merge their work.

*Adapted from [Solo work with Git](https://github.com/alan-turing-institute/rse-course/blob/main/module04_version_control_with_git/04_01_solo_work_with_git.ipynb) (The Alan Turing Institute, *Research Software Engineering with Python*, CC BY 3.0); local copy: [extras/17-solo-work-with-git.ipynb](extras/17-solo-work-with-git.ipynb).*

### Repositories

A **repository** is a project folder whose history Git keeps, in the hidden folder `.git`. Git is used from a terminal or from the Source Control panel of VS Code. Before the first commit on a computer, set your name and e-mail once: `git config --global user.name "Ada Example"` and `git config --global user.email "ada@example.org"`.

To show real Git output here, we create a repository in a temporary folder. The helper `git` runs one Git command in a repository. It sets the author's name and a fixed date for each commit, so that the hashes are the same every time the notebook runs, and it ignores your personal Git settings (`GIT_CONFIG_GLOBAL`).

In [20]:
CLOCK = iter(range(10_000))


def git(repo, *args, who="Ada"):
    """Run one git command in repo, as the person `who`, with a reproducible timestamp."""
    minute = next(CLOCK)
    stamp = f"2026-10-14T{9 + minute // 60:02d}:{minute % 60:02d}:00+02:00"
    env = {"GIT_CONFIG_GLOBAL": os.devnull, "GIT_CONFIG_NOSYSTEM": "1",
           "GIT_AUTHOR_NAME": who, "GIT_AUTHOR_EMAIL": f"{who.lower()}@example.org",
           "GIT_COMMITTER_NAME": who, "GIT_COMMITTER_EMAIL": f"{who.lower()}@example.org",
           "GIT_AUTHOR_DATE": stamp, "GIT_COMMITTER_DATE": stamp}
    run("git", *args, cwd=repo, env=env)


repo = Path(tempfile.mkdtemp(prefix="git-demo-"))
git(repo, "init", "--initial-branch=main")

Initialized empty Git repository in /private/var/folders/_2/pj2r648n4696ddzb0qltgpdr0000gn/T/git-demo-2ipzhilz/.git/


### Commits and the staging area

- A **commit** is a snapshot of all tracked files with an author, a date, a message and a unique identifier, the **hash** (shown shortened to seven characters). Each commit points to its parent, so the commits form a history.
- A change passes through three places: the **working directory** (the files you edit), the **staging area** or index (the changes selected for the next commit, `git add`) and the **repository** (`git commit`).

```mermaid
flowchart LR
  W["Working directory<br/>(files you edit)"] -- "git add" --> S["Staging area<br/>(the next commit)"]
  S -- "git commit" --> R[("Local repository<br/>(history in .git)")]
  R -- "git push" --> O[("Remote on GitHub<br/>(origin)")]
  O -- "git pull" --> R
```

Next, the first file: the rules of the listing analysis, kept in one place.

In [21]:
%%writefile {repo}/rules.py
"""Rules of the listing analysis, kept in one place."""

SHORT_STAY_MAX_NIGHTS = 28
PRICE_CAP = 1000

Writing /var/folders/_2/pj2r648n4696ddzb0qltgpdr0000gn/T/git-demo-2ipzhilz/rules.py


`git status --short` lists files and their state (`??` untracked); `git add` stages a file; `git commit -m` records the staged snapshot with a message; `git log --oneline` shows the history, one commit per line.

In [22]:
git(repo, "status", "--short")
git(repo, "add", "rules.py")
git(repo, "commit", "-m", "Add analysis rules")
git(repo, "log", "--oneline")

?? rules.py


[main (root-commit) 56451f6] Add analysis rules
 1 file changed, 4 insertions(+)
 create mode 100644 rules.py
56451f6 Add analysis rules


Commit messages are written in the imperative, as if completing "This commit will ...": *Add digit count*, *Fix empty-text bug in n_words*. One logical change per commit.

#### Staging selectively

Git commits only what was staged. This lets you split work into logical commits: if you changed two files for two reasons, stage and commit them separately. `git commit -a` skips the choice and stages every modified tracked file (but never new, untracked files).

In [23]:
(repo / "README.md").write_text("# Listing rules\n\nThe rules of the analysis live in rules.py.\n")
(repo / "rules.py").write_text((repo / "rules.py").read_text() + "MIN_REVIEWS = 1\n")
git(repo, "status", "--short")
git(repo, "add", "README.md")
git(repo, "commit", "-m", "Add README")
git(repo, "status", "--short")

 M rules.py
?? README.md


[main af1129e] Add README
 1 file changed, 3 insertions(+)
 create mode 100644 README.md


 M rules.py


Only the README was committed; `rules.py` is still modified (` M`) but unstaged. We commit it separately.

In [24]:
git(repo, "commit", "-am", "Add a minimum number of reviews")
git(repo, "log", "--oneline", "-3")

[main f2751bb] Add a minimum number of reviews
 1 file changed, 1 insertion(+)
f2751bb Add a minimum number of reviews
af1129e Add README
56451f6 Add analysis rules


#### History, hashes and `git show`

Each commit has a parent; the commits form a **graph**. The hash, such as `238eaff15e2769e0ef1d989f1a2e8be1873fa0ab`, identifies a commit uniquely in every copy of the repository; the first seven or so characters are usually enough. Branches, tags and `HEAD` are only labels pointing at commits, and some operations, such as a fast-forward, only move labels.

*Adapted from [Git theory](https://github.com/alan-turing-institute/rse-course/blob/main/module04_version_control_with_git/04_06_git_theory.ipynb) (The Alan Turing Institute, *Research Software Engineering with Python*, CC BY 3.0); local copy: [extras/19-git-theory.ipynb](extras/19-git-theory.ipynb).*

Next, the full log entry of the last commit, and `git show` for the change one commit made. `HEAD~1` means "the parent of HEAD".

In [25]:
git(repo, "log", "-1")
git(repo, "show", "--stat", "HEAD~1")

commit f2751bb29c1287f09d1090d4b7da042981f00f86
Author: Ada <ada@example.org>
Date:   Wed Oct 14 09:09:00 2026 +0200

    Add a minimum number of reviews


commit af1129effa6d3ef1ef0d62b3cee4fc5621277f3d
Author: Ada <ada@example.org>
Date:   Wed Oct 14 09:07:00 2026 +0200

    Add README

 README.md | 3 +++
 1 file changed, 3 insertions(+)


### Keeping files out of Git: `.gitignore`

A file `.gitignore` lists patterns of files that Git should never track: environments (`.venv/`), caches (`__pycache__/`), data (`*.parquet`, `data/`) and secrets (`.env`). Ignored files do not appear in `git status` and are not added by `git add .`.

> **Note.** Never commit data files, `.env` files, passwords or API keys. Git keeps every version: deleting a file in a later commit does not remove it from the history. Write the `.gitignore` before the first commit.

In [26]:
(repo / ".env").write_text("API_KEY=do-not-share\n")
(repo / "listings.parquet").write_bytes(b"not really parquet")
git(repo, "status", "--short")
(repo / ".gitignore").write_text(".venv/\n__pycache__/\n*.parquet\ndata/\n.env\n")
git(repo, "status", "--short")

?? .env
?? listings.parquet


?? .gitignore


In [27]:
git(repo, "add", ".")
git(repo, "commit", "-m", "Ignore environments, caches, data and secrets")
git(repo, "show", "--stat", "--oneline", "HEAD")

[main 320f18f] Ignore environments, caches, data and secrets
 1 file changed, 5 insertions(+)
 create mode 100644 .gitignore


320f18f Ignore environments, caches, data and secrets
 .gitignore | 5 +++++
 1 file changed, 5 insertions(+)


Only `.gitignore` was committed; the secret and the data stayed on the disk only.

### Branches

A **branch** is a movable label on a commit; new commits on the branch move the label forward. `main` holds the agreed version. **HEAD** marks the branch you are on.

Next a branch. `git switch -c name` creates a branch and moves to it. On the branch we change the short-stay rule to 30 nights.

In [28]:
git(repo, "switch", "-c", "feature/monthly-rule")

Switched to a new branch 'feature/monthly-rule'


In [29]:
%%writefile {repo}/rules.py
"""Rules of the listing analysis, kept in one place."""

SHORT_STAY_MAX_NIGHTS = 30
PRICE_CAP = 1000
MIN_REVIEWS = 1

Overwriting /var/folders/_2/pj2r648n4696ddzb0qltgpdr0000gn/T/git-demo-2ipzhilz/rules.py


`git diff` shows the change line by line: `-` marks a removed line, `+` an added one. `git commit -am` stages every modified tracked file and commits in one step.

In [30]:
git(repo, "diff")
git(repo, "commit", "-am", "Use 30 nights as the short-stay limit")

diff --git a/rules.py b/rules.py
index 496e18e..1035b1c 100644
--- a/rules.py
+++ b/rules.py
@@ -1,5 +1,5 @@
 """Rules of the listing analysis, kept in one place."""
 
-SHORT_STAY_MAX_NIGHTS = 28
+SHORT_STAY_MAX_NIGHTS = 30
 PRICE_CAP = 1000
 MIN_REVIEWS = 1


[feature/monthly-rule ad6ff40] Use 30 nights as the short-stay limit
 1 file changed, 1 insertion(+), 1 deletion(-)


### Merging: fast-forward and merge commits

`git merge <branch>` brings another branch's commits into the current one. If the current branch has not moved since the other started, Git just moves the label forward (**fast-forward**). Otherwise it creates a **merge commit** with two parents.

**Predict before running.** We go back to `main` and merge the branch. `main` has not moved since the branch started: what kind of merge is it?

In [31]:
git(repo, "switch", "main")
git(repo, "merge", "feature/monthly-rule")
git(repo, "log", "--oneline", "--graph", "--all")

Switched to branch 'main'


Updating 320f18f..ad6ff40
Fast-forward
 rules.py | 2 +-
 1 file changed, 1 insertion(+), 1 deletion(-)
* ad6ff40 Use 30 nights as the short-stay limit
* 320f18f Ignore environments, caches, data and secrets
* f2751bb Add a minimum number of reviews
* af1129e Add README
* 56451f6 Add analysis rules


A **fast-forward**: Git only moved the `main` label to the branch's commit.

### Merge conflicts

A **merge conflict** occurs when both branches changed the same lines differently. Git stops, writes both versions into the file between **conflict markers**, and a person decides.

Next, a conflict. Ada lowers the price cap on a branch; meanwhile Ben, on `main`, raises it. Both change the same line.

In [32]:
git(repo, "switch", "-c", "fix/price-cap")

Switched to a new branch 'fix/price-cap'


In [33]:
%%writefile {repo}/rules.py
"""Rules of the listing analysis, kept in one place."""

SHORT_STAY_MAX_NIGHTS = 30
PRICE_CAP = 500
MIN_REVIEWS = 1

Overwriting /var/folders/_2/pj2r648n4696ddzb0qltgpdr0000gn/T/git-demo-2ipzhilz/rules.py


In [34]:
git(repo, "commit", "-am", "Lower the price cap to 500")
git(repo, "switch", "main")

[fix/price-cap 8383fb5] Lower the price cap to 500
 1 file changed, 1 insertion(+), 1 deletion(-)


Switched to branch 'main'


In [35]:
%%writefile {repo}/rules.py
"""Rules of the listing analysis, kept in one place."""

SHORT_STAY_MAX_NIGHTS = 30
PRICE_CAP = 2000
MIN_REVIEWS = 1

Overwriting /var/folders/_2/pj2r648n4696ddzb0qltgpdr0000gn/T/git-demo-2ipzhilz/rules.py


In [36]:
git(repo, "commit", "-am", "Raise the price cap to 2000", who="Ben")
git(repo, "log", "--oneline", "--graph", "--all")

[main d2c8664] Raise the price cap to 2000
 1 file changed, 1 insertion(+), 1 deletion(-)
* d2c8664 Raise the price cap to 2000
| * 8383fb5 Lower the price cap to 500
|/  
* ad6ff40 Use 30 nights as the short-stay limit
* 320f18f Ignore environments, caches, data and secrets
* f2751bb Add a minimum number of reviews
* af1129e Add README
* 56451f6 Add analysis rules


The history has forked: both branches have a commit the other lacks. **Predict before running:** what will `git merge fix/price-cap` do, and what will `rules.py` contain afterwards?

In [37]:
git(repo, "merge", "fix/price-cap")
git(repo, "status", "--short")
print((repo / "rules.py").read_text())

Auto-merging rules.py
CONFLICT (content): Merge conflict in rules.py
Automatic merge failed; fix conflicts and then commit the result.


UU rules.py
"""Rules of the listing analysis, kept in one place."""

SHORT_STAY_MAX_NIGHTS = 30
<<<<<<< HEAD
PRICE_CAP = 2000
PRICE_CAP = 500
>>>>>>> fix/price-cap
MIN_REVIEWS = 1



Git merged nothing in the conflicting line and asks us to decide. Between `<<<<<<< HEAD` and `=======` is the version of the current branch (`main`, Ben's 2000); between `=======` and `>>>>>>> fix/price-cap` the incoming version (Ada's 500). `UU` in the status means "both modified, unmerged". Lines outside the markers merged without problems.

A conflict is a question for the people involved, not an error of Git. Ada and Ben talk and agree on 800. We write the resolved file without markers, stage it to mark the conflict as resolved, and commit the merge.

In [38]:
%%writefile {repo}/rules.py
"""Rules of the listing analysis, kept in one place."""

SHORT_STAY_MAX_NIGHTS = 30
PRICE_CAP = 800  # agreed by Ada and Ben: 500 removed real flats, 2000 kept data errors
MIN_REVIEWS = 1

Overwriting /var/folders/_2/pj2r648n4696ddzb0qltgpdr0000gn/T/git-demo-2ipzhilz/rules.py


In [39]:
git(repo, "add", "rules.py")
git(repo, "commit", "-m", "Merge fix/price-cap: agree on a cap of 800")
git(repo, "log", "--oneline", "--graph", "--all")

[main 1a969e9] Merge fix/price-cap: agree on a cap of 800


*   1a969e9 Merge fix/price-cap: agree on a cap of 800
|\  
| * 8383fb5 Lower the price cap to 500
* | d2c8664 Raise the price cap to 2000
|/  
* ad6ff40 Use 30 nights as the short-stay limit
* 320f18f Ignore environments, caches, data and secrets
* f2751bb Add a minimum number of reviews
* af1129e Add README
* 56451f6 Add analysis rules


The merge commit has two parents, drawn as two lines joining. If you get lost during a merge, `git merge --abort` returns to the state before it. VS Code shows buttons (*Accept Current*, *Accept Incoming*, *Accept Both*) above each conflict, and a merge editor; check the result by eye anyway.

### Comparing branches

`git log A..B` lists the commits that are reachable from `B` but not from `A`: "what is on B that A does not have yet". `git log --left-right A...B` (three dots) lists the commits on either side since the two diverged, marked `<` and `>`. `git diff A...B` shows the changes on `B` since it left `A`, which is exactly what a pull request shows (see the section on GitHub).

*Adapted from [Branches](https://github.com/alan-turing-institute/rse-course/blob/main/module04_version_control_with_git/04_07_branches.ipynb) (The Alan Turing Institute, *Research Software Engineering with Python*, CC BY 3.0); local copy: [extras/18-git-branches.ipynb](extras/18-git-branches.ipynb).*

In [40]:
git(repo, "switch", "-c", "feature/min-price")
(repo / "rules.py").write_text((repo / "rules.py").read_text() + "MIN_PRICE = 10\n")
git(repo, "commit", "-am", "Add a minimum price")
git(repo, "switch", "main")
(repo / "README.md").write_text((repo / "README.md").read_text() + "\nPrices are in EUR per night.\n")
git(repo, "commit", "-am", "Document the price unit", who="Ben")

Switched to a new branch 'feature/min-price'
[feature/min-price eda83f8] Add a minimum price
 1 file changed, 1 insertion(+)


Switched to branch 'main'


[main fdb56ba] Document the price unit
 1 file changed, 2 insertions(+)


In [41]:
print("on the branch, not on main:")
git(repo, "log", "--oneline", "main..feature/min-price")
print("on either side since they diverged:")
git(repo, "log", "--oneline", "--left-right", "main...feature/min-price")

on the branch, not on main:


eda83f8 Add a minimum price
on either side since they diverged:
< fdb56ba Document the price unit
> eda83f8 Add a minimum price


A branch strategy that many teams use: `main` holds the working version; short-lived **feature branches** hold one idea each; some projects add a `develop` branch for integration and `release` branches for versions shared with others. Two related commands: `git restore --source=<branch> <file>` takes one file's content from another branch, and `git cherry-pick <commit>` applies one commit from elsewhere to the current branch.

### Undoing things

Git can undo a change at each of its three places: in the working directory, in the staging area and in the history.

| Situation | Command | Effect |
|---|---|---|
| An edit in the working directory that you want to throw away | `git restore <file>` | the file returns to its last committed (or staged) state |
| A file staged by mistake | `git restore --staged <file>` | unstaged again; the edit stays |
| A commit that is already shared | `git revert <commit>` | a *new* commit that undoes it; history is not rewritten |
| Your last local commit, not yet pushed | `git reset --soft HEAD~1` | the branch label moves back; the changes stay staged |
| | `git reset HEAD~1` (`--mixed`) | label and staging area move back; the changes stay in the files |
| | `git reset --hard HEAD~1` | label, staging area and files all move back: the changes are gone |

Never rewrite history (`reset`, `rebase`) that others have already pulled; use `revert` instead. Next, a mistake committed and pushed, then undone with `revert`.

*Adapted from [Git theory](https://github.com/alan-turing-institute/rse-course/blob/main/module04_version_control_with_git/04_06_git_theory.ipynb) (The Alan Turing Institute, *Research Software Engineering with Python*, CC BY 3.0); local copy: [extras/19-git-theory.ipynb](extras/19-git-theory.ipynb).*

In [42]:
(repo / "rules.py").write_text((repo / "rules.py").read_text().replace("PRICE_CAP = 800", "PRICE_CAP = 8"))
git(repo, "commit", "-am", "Lower the price cap")          # a typo: 8 instead of 800
git(repo, "revert", "--no-edit", "HEAD")
git(repo, "log", "--oneline", "-3")
print((repo / "rules.py").read_text())

[main f7c87e2] Lower the price cap
 1 file changed, 1 insertion(+), 1 deletion(-)


[main 84856ff] Revert "Lower the price cap"
 Date: Wed Oct 14 09:41:00 2026 +0200
 1 file changed, 1 insertion(+), 1 deletion(-)
84856ff Revert "Lower the price cap"
f7c87e2 Lower the price cap
fdb56ba Document the price unit
"""Rules of the listing analysis, kept in one place."""

SHORT_STAY_MAX_NIGHTS = 30
PRICE_CAP = 800  # agreed by Ada and Ben: 500 removed real flats, 2000 kept data errors
MIN_REVIEWS = 1



### Remotes and collaboration: a rejected push

A **remote** is another copy of the repository, usually on GitHub, called `origin`. `git clone <url>` copies a repository once; `git push` uploads your commits; `git pull` downloads new commits and merges them into your branch. Git refuses a push that would overwrite commits you do not have yet: you must pull first, which may create a merge commit (or a conflict, as above).

To try this without GitHub, a **bare** repository in a folder plays the role of GitHub, and two clones play Ada's and Ben's laptops. (The same works for a team with a shared folder or a USB stick.)

*Adapted from [Collaboration](https://github.com/alan-turing-institute/rse-course/blob/main/module04_version_control_with_git/04_04_collaboration.ipynb) (The Alan Turing Institute, *Research Software Engineering with Python*, CC BY 3.0); local copy: [extras/20-git-collaboration.ipynb](extras/20-git-collaboration.ipynb).* The original uses a real GitHub repository and two people.

In [43]:
team = Path(tempfile.mkdtemp(prefix="git-team-"))
git(team, "init", "--bare", "--initial-branch=main", "origin.git")
git(team, "clone", "origin.git", "ada")
git(team, "clone", "origin.git", "ben", who="Ben")
ada, ben = team / "ada", team / "ben"
git(ben, "config", "pull.rebase", "false", who="Ben")   # pull = fetch + merge

Initialized empty Git repository in /private/var/folders/_2/pj2r648n4696ddzb0qltgpdr0000gn/T/git-team-fzf5tq1r/origin.git/


Cloning into 'ada'...
done.


Cloning into 'ben'...
done.


In [44]:
(ada / "guide.md").write_text("# Berlin weekend guide\n\n## Food\n\n## Sights\n")
git(ada, "add", "guide.md")
git(ada, "commit", "-m", "Start the guide")
git(ada, "push", "-u", "origin", "main")
git(ben, "pull", who="Ben")

[main (root-commit) 3232900] Start the guide
 1 file changed, 5 insertions(+)
 create mode 100644 guide.md


branch 'main' set up to track 'origin/main'.
To /private/var/folders/_2/pj2r648n4696ddzb0qltgpdr0000gn/T/git-team-fzf5tq1r/origin.git
 * [new branch]      main -> main


From /private/var/folders/_2/pj2r648n4696ddzb0qltgpdr0000gn/T/git-team-fzf5tq1r/origin
 * [new branch]      main       -> origin/main


Both now have the same history. Each changes a *different* file and commits. Ada pushes first. **Predict before running:** what happens to Ben's push?

In [45]:
(ada / "food.md").write_text("Best Döner: Rüyam\n")
git(ada, "add", "food.md"); git(ada, "commit", "-m", "Add food tips"); git(ada, "push")
(ben / "sights.md").write_text("Tempelhofer Feld\n")
git(ben, "add", "sights.md", who="Ben"); git(ben, "commit", "-m", "Add sights", who="Ben")
git(ben, "push", who="Ben")

[main f9210e7] Add food tips
 1 file changed, 1 insertion(+)
 create mode 100644 food.md


To /private/var/folders/_2/pj2r648n4696ddzb0qltgpdr0000gn/T/git-team-fzf5tq1r/origin.git
   3232900..f9210e7  main -> main


[main d60d7e3] Add sights
 1 file changed, 1 insertion(+)
 create mode 100644 sights.md
To /private/var/folders/_2/pj2r648n4696ddzb0qltgpdr0000gn/T/git-team-fzf5tq1r/origin.git
 ! [rejected]        main -> main (fetch first)
error: failed to push some refs to '/private/var/folders/_2/pj2r648n4696ddzb0qltgpdr0000gn/T/git-team-fzf5tq1r/origin.git'
hint: Updates were rejected because the remote contains work that you do not
hint: have locally. This is usually caused by another repository pushing to
hint: the same ref. If you want to integrate the remote changes, use
hint: 'git pull' before pushing again.
hint: See the 'Note about fast-forwards' in 'git push --help' for details.


Rejected: the remote has Ada's commit, which Ben does not have. Ben pulls; the changes touch different files, so Git merges them automatically with a merge commit, and the push succeeds.

In [46]:
git(ben, "pull", "--no-edit", who="Ben")
git(ben, "push", who="Ben")
git(ben, "log", "--oneline", "--graph")

Merge made by the 'ort' strategy.
 food.md | 1 +
 1 file changed, 1 insertion(+)
 create mode 100644 food.md
From /private/var/folders/_2/pj2r648n4696ddzb0qltgpdr0000gn/T/git-team-fzf5tq1r/origin
   3232900..f9210e7  main       -> origin/main
To /private/var/folders/_2/pj2r648n4696ddzb0qltgpdr0000gn/T/git-team-fzf5tq1r/origin.git
   f9210e7..f5a695f  main -> main
*   f5a695f Merge branch 'main' of /private/var/folders/_2/pj2r648n4696ddzb0qltgpdr0000gn/T/git-team-fzf5tq1r/origin
|\  
| * f9210e7 Add food tips
* | d60d7e3 Add sights
|/  
* 3232900 Start the guide


Had both changed the same line, the pull would have stopped with a conflict, resolved exactly as before. Habits that reduce conflicts: pull before you start, keep branches short-lived and small, and announce large restructurings (renaming or deleting files) before doing them.

> 📖 **Read more:** [extras/16-version-control-introduction.ipynb](extras/16-version-control-introduction.ipynb), [extras/17-solo-work-with-git.ipynb](extras/17-solo-work-with-git.ipynb), [extras/18-git-branches.ipynb](extras/18-git-branches.ipynb), [extras/19-git-theory.ipynb](extras/19-git-theory.ipynb) and [extras/20-git-collaboration.ipynb](extras/20-git-collaboration.ipynb) (Turing RSE course): the same steps on a text project, with the original outputs.

## GitHub: pull requests and code review

GitHub adds two ways of working together on top of Git: pull requests and code review. In the final project, each member's pull requests and reviews are part of the assessment.

### Pull requests

A **pull request** (PR) on GitHub proposes to merge a branch into `main`. It shows the **diff** (every changed line, in the *Files changed* tab), the results of automated checks, and a conversation. The pull request is where a second person checks a change before it reaches `main`, and it spreads knowledge: at least two people know every part of the code.

```mermaid
flowchart LR
  B["Branch"] --> C["Commits"] --> P["git push"] --> PR["Pull request"]
  PR --> CI{"CI checks<br/>green?"}
  PR --> R{"Review<br/>approved?"}
  CI -- no --> C
  R -- "changes requested" --> C
  CI -- yes --> M["Merge into main"]
  R -- yes --> M
```

The commands on the author's side:

```bash
git switch -c feature/min-price
# ... implement and test: uv run pytest -q; uv run ruff check .
git commit -am "Add a minimum price"
git push -u origin feature/min-price     # -u: remember origin as this branch's upstream
# GitHub prints a link: open the pull request, describe what and why, request a reviewer
# after review: fix, commit, git push (the PR updates); merge when approved and CI is green
git switch main && git pull              # bring your local main up to date
```

Next, what the reviewer sees in *Files changed* for the branch `feature/min-price` of our demo repository: `git diff main...feature/min-price` (three dots) shows the changes made on the branch since it left `main`, ignoring what happened on `main` meanwhile.

In [47]:
git(repo, "diff", "main...feature/min-price")

diff --git a/rules.py b/rules.py
index 1589fc3..0e17ebd 100644
--- a/rules.py
+++ b/rules.py
@@ -3,3 +3,4 @@
 SHORT_STAY_MAX_NIGHTS = 30
 PRICE_CAP = 800  # agreed by Ada and Ben: 500 removed real flats, 2000 kept data errors
 MIN_REVIEWS = 1
+MIN_PRICE = 10


#### Describing a pull request and linking issues

A PR description that reviewers appreciate says what, why and how it was tested:

```markdown
## What
Adds `MIN_PRICE = 10` to `rules.py`.

## Why
Listings below 10 EUR per night are placeholders, not real offers. Closes #4.

## How tested
New test with a 9 EUR and a 10 EUR listing; `uv run pytest -q` passes locally.
```

`Closes #4` links the PR to issue 4 and closes the issue when the PR is merged. An **issue** describes a task or a bug; GitHub numbers issues and PRs together.

#### Pull-request states

A pull request is in one state at a time. Open a **draft** PR early to show work in progress; mark it *ready for review* when it is. The states and the events that move a PR between them:

```mermaid
stateDiagram-v2
  [*] --> Draft: open as draft
  Draft --> Open: ready for review
  Open --> ChangesRequested: reviewer comments
  ChangesRequested --> Open: author pushes fixes
  Open --> Approved: reviewer approves
  Approved --> Merged: CI green, merge
  Open --> Closed: not wanted
  Merged --> [*]
```

### Fork and pull

In a team repository, members are **collaborators** with permission to push branches. To contribute to a project where you have no write permission, such as NumPy, you **fork** it: GitHub makes a copy in your account, where you can push. The workflow is then:

1. **Fork** the repository on GitHub (the *Fork* button).
2. **Clone** your fork: `git clone git@github.com:<you>/<project>.git`.
3. Create a **feature branch**: `git switch -c fix-median`.
4. Make, commit and **push** the change to your fork: `git push origin fix-median`.
5. On GitHub, *Compare & pull request*: describe the change and create the PR against the original repository.
6. The maintainers review it; in *Files changed* they hover over a line and click `+` to comment on it.
7. You fix, commit and push to the same branch; the PR updates.
8. A maintainer merges it.

Forks and pull requests exist only on the server (GitHub); there is no `git fork` command. Before starting, read the project's contribution guide, and look at existing issues and pull requests: someone may already be working on the same thing.

*Adapted from [Fork and Pull](https://github.com/alan-turing-institute/rse-course/blob/main/module04_version_control_with_git/04_05_fork_and_pull.ipynb) (The Alan Turing Institute, *Research Software Engineering with Python*, CC BY 3.0); local copy: [extras/21-fork-and-pull.ipynb](extras/21-fork-and-pull.ipynb).*

### Code review

In a **code review**, a teammate reads the change, comments on specific lines, and either **approves** it or **requests changes**; the author pushes fixes to the same branch and the PR updates itself.

What a reviewer checks, in this order:

1. **Purpose.** Does the PR do what its description says, and only that?
2. **Correctness.** Edge cases, error handling, off-by-one errors, leakage of test data into training.
3. **Tests.** Is the new behaviour tested? Would the tests fail if the code were wrong?
4. **Readability.** Names, docstrings, type hints, no dead code.
5. **Safety.** No data files, keys or large notebooks with outputs.

#### Review etiquette

- Keep pull requests small: one feature or fix. Large PRs get superficial reviews.
- Review the code, not the person: "This fails for an empty title; could we return 0?" rather than "You forgot the empty case".
- As an author, answer every comment, by a change or a short explanation.

In practice, Google's public engineering guide to code review sets the standard that a reviewer should approve a change once it clearly improves the overall health of the code, even if it is not perfect, and it recommends small changes. CPython and scikit-learn accept changes only through reviewed pull requests with passing CI.

## Continuous integration with GitHub Actions

**Continuous integration** (CI) means that every proposed change is merged early and checked automatically in a clean environment. It removes "it works on my machine": the checks run the same way for everybody, on a machine that has only what the lock file specifies.

### "It works on my machine"

Code can behave differently on different computers: another operating system, another Python version, another package version, a file that exists only on your disk. The tests below are deliberately written so that their result depends on the machine: the first passes only where the platform name contains a digit (`win32`), the second depends on a detail of `str.replace` that differs between Python versions. Run them here and see which pass on this computer.

*Adapted from [Continuous Integration](https://github.com/alan-turing-institute/rse-course/blob/main/module05_testing_your_code/05_06_continuous_integration.ipynb) (The Alan Turing Institute, *Research Software Engineering with Python*, CC BY 3.0); local copy: [extras/22-continuous-integration.ipynb](extras/22-continuous-integration.ipynb).*

In [48]:
ci_demo = Path(tempfile.mkdtemp(prefix="ci-demo-"))
(ci_demo / "test_demo.py").write_text(
    "import re\nimport sys\n\n\n"
    "def test_platform():\n    assert re.search(r'\\d', sys.platform)\n\n\n"
    "def test_replace():\n    assert ''.replace('', 'A', 2) == 'A'\n"
)
run(sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "--no-header", "--tb=line", cwd=ci_demo)
print("this machine:", sys.platform, "Python", sys.version.split()[0])

F.                                                                       [100%]
=================================== FAILURES ===================================
E   AssertionError: assert None
     +  where None = <function search at 0x10a27fcc0>('\\d', 'darwin')
     +    where <function search at 0x10a27fcc0> = re.search
     +    and   'darwin' = sys.platform
/private/var/folders/_2/pj2r648n4696ddzb0qltgpdr0000gn/T/ci-demo-8htem5oe/test_demo.py:6: AssertionError: assert None
=========================== short test summary info ============================
FAILED test_demo.py::test_platform - AssertionError: assert None
1 failed, 1 passed in 0.02s
this machine: darwin Python 3.14.7


A CI server runs the tests on a clean, defined machine, and can run them on several. The next subsections show how.

### Workflows in GitHub Actions

**GitHub Actions** runs **workflows** described in YAML files under `.github/workflows/` at the root of a repository.

- A workflow has **triggers** (`on:`), for example every pull request and every push to `main`.
- It has one or more **jobs**; each runs on a fresh virtual machine (`runs-on: ubuntu-latest`).
- A job is a list of **steps**: a ready-made **action** (`uses:`, such as `actions/checkout`) or a shell command (`run:`).
- If any step fails, the run is marked red on the pull request.

Next, the workflow of the workspace, [`.github/workflows/ci.yml`](../workspace/.github/workflows/ci.yml), printed from the file. Read it from the top: name, triggers, one job called `test`, and its steps.

In [49]:
print((WORKSPACE / ".github" / "workflows" / "ci.yml").read_text())

# Continuous integration: lint and test every pull request and every push to main.
# GitHub reads workflows only from .github/workflows/ at the root of a repository:
# copy this workspace into its own repository (or your team repository) first.
name: CI

on:
  pull_request:
  push:
    branches: [main]

jobs:
  test:                                   # job name = the required status check in branch protection
    runs-on: ubuntu-latest                # a fresh virtual machine for every run
    steps:
      - uses: actions/checkout@v7.0.1         # copy the repository onto the machine
      - uses: astral-sh/setup-uv@v10.3.0  # install uv; Python comes from .python-version
        with:
          enable-cache: true
      - name: Install the locked environment
        run: uv sync --locked             # fails if uv.lock does not match pyproject.toml
      - name: Lint
        run: uv run ruff check .
      - name: Check formatting
        run: uv run ruff format --check .
      - name: R

`uv sync --locked` fails if `uv.lock` does not match `pyproject.toml`, so a forgotten lock-file update is caught too.

YAML is indentation-sensitive and uses spaces, never tabs; GitHub shows errors in a workflow file in the *Actions* tab. Workflows are only read from `.github/workflows/` at the root of the repository, so the workspace's file becomes active once the workspace is its own repository (exercise 2.4).

In practice, pandas, scikit-learn and FastAPI run their tests and linters in GitHub Actions on every pull request; a red check blocks the merge.

#### Test matrices

A **matrix** repeats a job for each combination of values; `fail-fast: false` lets all combinations finish even if one fails:

```yaml
jobs:
  test:
    strategy:
      fail-fast: false
      matrix:
        os: [ubuntu-latest, windows-latest, macos-latest]
        python-version: ["3.13", "3.14"]
    runs-on: ${{ matrix.os }}
    steps:
      - uses: actions/checkout@v7.0.1
      - uses: astral-sh/setup-uv@v10.3.0
        with:
          python-version: ${{ matrix.python-version }}
      - run: uv sync --locked
      - run: uv run pytest -q
```

Here `${{ matrix.os }}` is replaced by each value in turn: six jobs in all. pytest can skip tests that only make sense on one platform (`@pytest.mark.skipif(sys.platform == "win32", reason=...)`). The tool `act` runs GitHub Actions workflows locally, which helps when debugging a workflow.

### Linting and formatting with ruff

A **linter** reads code without running it and reports likely errors and breaks of coding conventions (PEP 8 is Python's style guide). A **formatter** rewrites the layout of the code (spaces, line breaks, quotes) into one standard style, so that diffs show only real changes. The course uses **ruff** for both; it replaces the older trio flake8 (errors and style), black (formatting) and isort (import order). Style guides are guides, not laws: do not blindly accept every automated suggestion.

*Adapted from [Linting](https://github.com/alan-turing-institute/rse-course/blob/main/module07_construction_and_design/07_03_linters.ipynb) (The Alan Turing Institute, *Research Software Engineering with Python*, CC BY 3.0); local copy: [extras/23-linting.ipynb](extras/23-linting.ipynb).* The original demonstrates flake8, black and isort; this version uses ruff, which implements the same rules.

Next, a file with an unused import, a missing blank line and an undefined name. **Predict:** which problems will ruff report?

In [50]:
%%writefile {ci_demo}/geometry.py

from math import e

def circumference(r):
    return 2 * pi * r

Writing /var/folders/_2/pj2r648n4696ddzb0qltgpdr0000gn/T/ci-demo-8htem5oe/geometry.py


In [51]:
run(sys.executable, "-m", "ruff", "check", "--select", "E,F", "--preview", "--output-format", "concise", "geometry.py", cwd=ci_demo)

geometry.py:2:18: unused-import: [*] `math.e` imported but unused
geometry.py:4:1: blank-lines-top-level: [*] Expected 2 blank lines, found 1
geometry.py:5:16: undefined-name: Undefined name `pi`
Found 3 errors.
[*] 2 fixable with `--fix`.


F401 (unused import) and F821 (undefined name `pi`) are likely bugs; E302 (two blank lines expected before a function) is style. `ruff check --fix` repairs some problems automatically (it removes the unused import, for example). Next, the formatter: `ruff format --diff` shows what it would change without changing the file.

In [52]:
%%writefile {ci_demo}/messy.py
import numpy as np
def my_complex_function(important_argument_1,important_argument_2,optional_argument_3 = 2):
    result = np.array([important_argument_1,important_argument_2])*optional_argument_3
    print( 'result:' , result )
    return result

Writing /var/folders/_2/pj2r648n4696ddzb0qltgpdr0000gn/T/ci-demo-8htem5oe/messy.py


In [53]:
run(sys.executable, "-m", "ruff", "format", "--diff", "messy.py", cwd=ci_demo)

--- messy.py
+++ messy.py
@@ -1,5 +1,11 @@
 import numpy as np
-def my_complex_function(important_argument_1,important_argument_2,optional_argument_3 = 2):
-    result = np.array([important_argument_1,important_argument_2])*optional_argument_3
-    print( 'result:' , result )
+
+
+def my_complex_function(
+    important_argument_1, important_argument_2, optional_argument_3=2
+):
+    result = (
+        np.array([important_argument_1, important_argument_2]) * optional_argument_3
+    )
+    print("result:", result)
     return result

1 file would be reformatted


The formatter added blank lines around the function, spaces around operators and after commas, removed spaces inside the `print` call, wrapped the long line, and used double quotes everywhere. The rule set `I` sorts imports in three groups: standard library, third-party, local.

In [54]:
%%writefile {ci_demo}/imports.py
import pandas as pd
import os
from matplotlib import pyplot as plt
import geometry
import sys

print(os, sys, pd, plt, geometry)

Writing /var/folders/_2/pj2r648n4696ddzb0qltgpdr0000gn/T/ci-demo-8htem5oe/imports.py


In [55]:
run(sys.executable, "-m", "ruff", "check", "--select", "I", "--diff", "imports.py", cwd=ci_demo)

--- imports.py
+++ imports.py
@@ -1,7 +1,9 @@
+import os
+import sys
+
 import pandas as pd
-import os
 from matplotlib import pyplot as plt
+
 import geometry
-import sys
 
 print(os, sys, pd, plt, geometry)

Would fix 1 error.


Settings live in `pyproject.toml` (the workspace selects the rule sets `E`, `F`, `I`, `B` and `UP` and allows lines of 100 characters). A warning you have a good reason to ignore can be silenced on one line with a comment such as `# noqa: F401`. Run linters often: editors such as VS Code show ruff's warnings while you type, the tool `pre-commit` runs them before every commit, and CI runs them on every pull request.

Run the same checks locally before you push, so that CI rarely surprises you:

```bash
uv run ruff check .          # likely errors and style rules; ruff check --fix repairs many
uv run ruff format .         # rewrite files into the standard layout
uv run pytest -q
```

### Branch protection

**Branch protection** (on GitHub: *Settings → Rules → Rulesets*) makes the checks mandatory for `main`: changes only through a pull request, at least one approving review, the CI job as a required **status check**, no force pushes. Broken code then cannot reach `main`.

```mermaid
sequenceDiagram
  participant D as Developer
  participant G as GitHub
  participant R as Runner (fresh VM)
  D->>G: git push, open pull request
  G->>R: start workflow "CI" (on: pull_request)
  R->>R: checkout, install uv, uv sync --locked
  R->>R: ruff check, ruff format --check, pytest
  R-->>G: status check "test": passed or failed
  G-->>D: green or red mark on the pull request
  Note over G: ruleset: merge only if the check passed and a review approved
```

### Notebooks and secrets in CI

Notebooks saved with outputs make large diffs and frequent conflicts. Clear outputs before committing exploratory notebooks (*Edit → Clear Outputs of All Cells*), or use a tool such as `nbstripout` that does it automatically. A workflow that needs a password or API key reads it from an encrypted **repository secret** (*Settings → Secrets and variables → Actions*), never from a file in the repository.

> 📖 **Read more:** [extras/22-continuous-integration.ipynb](extras/22-continuous-integration.ipynb) and [extras/23-linting.ipynb](extras/23-linting.ipynb) (Turing RSE course): a CI matrix on Windows and Linux, and flake8, black, isort, mypy and pylint side by side.

## Worked example · Does the listing package pass the checks CI would run?

We run the three checks of the workspace's workflow, ruff, the formatter check and pytest, on a fresh copy of the workspace in a temporary folder, as the CI runner would on a fresh machine, and then run the command-line program on the real listings. `shutil.copytree` copies the folder; `ignore_patterns` leaves out the local environment and caches.

In [56]:
ci_copy = Path(tempfile.mkdtemp(prefix="ci-run-")) / "workspace"
shutil.copytree(WORKSPACE, ci_copy,
                ignore=shutil.ignore_patterns(".venv", "__pycache__", ".pytest_cache", ".ruff_cache"))
run(sys.executable, "-m", "ruff", "check", ".", cwd=ci_copy)
run(sys.executable, "-m", "ruff", "format", "--check", ".", cwd=ci_copy)

All checks passed!


17 files already formatted


Next the tests. On the runner, `uv sync --locked` installs the package; here `PYTHONPATH=src` makes it importable instead. **Predict:** the workspace contains the exercises' tests marked `skip`. How will pytest report them?

In [57]:
run(sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "--no-header", cwd=ci_copy,
    env={"PYTHONPATH": "src"})

.........sss..........................................sssssssss......... [ 78%]
..........ssssssssss                                                     [100%]
70 passed, 22 skipped in 0.22s


All checks pass; the skipped tests are the exercises, which become active when you delete their `skip` lines. Finally the command-line program on the real data, run from the repository root as the README describes (the default path of the data is relative to the root).

In [58]:
run(sys.executable, "-m", "listingtools", env={"PYTHONPATH": str(ci_copy / "src")})

ListingTable(10221 listings)
{'n_listings': 10221,
 'n_hosts': 6660,
 'listings_per_district': {'Mitte': 2268,
                           'Friedrichshain-Kreuzberg': 2119,
                           'Pankow': 1562,
                           'Charlottenburg-Wilm.': 1146,
                           'Neukölln': 1046},
 'room_type_shares': {'Entire home/apt': 0.692,
                      'Private room': 0.294,
                      'Hotel room': 0.007,
                      'Shared room': 0.007},
 'n_short_stays': 5302,
 'median_short_stay_price': 157.5,
 'median_price_by_room_type': {'Entire home/apt': 185.0,
                               'Hotel room': 182.0,
                               'Private room': 98.0,
                               'Shared room': 54.0}}

Short-stay prices by room type (EUR per night):
                 n_listings  median_price
room_type                                
Entire home/apt        3694         185.0
Hotel room               63         182.0
Private ro

The same numbers as the notebook-style file, now from code that is versioned, linted, tested and runnable by anyone with one command. In the exercises you extend this package on branches, through reviewed pull requests, with CI.

## Check your understanding

1. Name two ways in which a notebook result can depend on hidden state, and one way to avoid each.
2. Why does program-only work go into `main()` under `if __name__ == "__main__":`?
3. What is the difference between `git add` and `git commit`? When does `git merge` produce a fast-forward, and when a merge commit?
4. You see `<<<<<<< HEAD` in a file. What do the parts above and below `=======` contain, and how do you finish the merge?
5. Why does the CI workflow run `uv sync --locked`, and what does branch protection add to CI?

## Exercises (30 min)

Work in pairs. The lecturer chooses which exercises to set; there are more than fit into 30 minutes, and *Stretch* tasks are for pairs that finish early. Exercises marked *workspace* are solved in the files of the [workspace](../workspace/README.md) (`cd sessions/02-software-engineering/workspace`, then `uv run pytest -q`). Exercises 2.4 and 2.5 need a GitHub account. You can run Git in a terminal, or in this notebook with the helper `git(folder, ...)` from the Git section.

### 2.1 · The shared shopping list (Git)

*Pairs · 15 min*

> Ana and Ben keep their flat's shopping list in Git. Starting from a list with three lines, Ana creates
> a branch and changes line 2 from `milk` to `oat milk`; Ben, on `main`, changes the same line to `2 milk`. Both
> commit. Merge Ana's branch into `main`. Before you press Enter: draw the commit graph and write down what the file
> will look like.

*Source:* [Pro Git, §3.2 Basic branching and merging](https://git-scm.com/book/en/v2/Git-Branching-Basic-Branching-and-Merging)

*Your answer (graph and predicted file):*

### 2.2 · Notebook against module (workspace)

*Paper · 10 min*

> Open `notebooks/explore_listings.py` and `src/listingtools/listings.py` side by side. List three differences (state, paths, testability) and note which of the five questions of the Session 1 case study each method of `ListingTable` answers. Which line would you have to change in each version if the short-stay rule became "fewer than 30 nights"?

*Source:* original exercise for this course (workspace exercise 3).

*Your answer:*

### 2.3 · Price by district and reviews (workspace)

*Code · 20 min*

> 1. Implement `ListingTable.median_price_by_district` (see the TODO in `src/listingtools/listings.py`). Delete the `skip` line above `test_median_price_by_district` in `tests/test_listings_exercises.py` and run `uv run pytest -q`.
> 2. Implement `review_summary` and activate its test in the same way. Why is the median number of reviews so much lower than the mean?
> 3. Add both results to `summary()`, activate the last test and run `uv run --project sessions/02-software-engineering/workspace listing-summary` from the repository root. Compare the numbers with the Session 1 case study.

*Source:* original exercise for this course (workspace exercises 4–6), on the Inside Airbnb Berlin snapshot of 26 June 2026 (CC BY 4.0).

*Your answer:*

### 2.4 · A pull request with a merge conflict (workspace)

*Pairs · 30 min*

> Copy the workspace into a new GitHub repository and push `main`. Person A creates the branch `feature/n-digits`, person B the branch `feature/upper-share`. Each implements one function in `src/listingtools/text_features.py`, adds its name to `FEATURE_NAMES`, deletes the `skip` line of its test in `tests/test_text_features.py`, adds one test of their own, commits, pushes and opens a pull request. The partner reviews it with at least one comment on a line; the author answers or fixes; the reviewer approves. Merge when CI is green. The second pull request will then have a merge conflict in the `FEATURE_NAMES` line: resolve it, keeping both names.

*Source:* original exercise for this course (workspace exercise 7); background: [Pro Git, §3.2](https://git-scm.com/book/en/v2/Git-Branching-Basic-Branching-and-Merging) and [GitHub Docs: Resolving a merge conflict using the command line](https://docs.github.com/en/pull-requests/collaborating-with-pull-requests/addressing-merge-conflicts/resolving-a-merge-conflict-using-the-command-line).

*Your answer:*

### 2.5 · The team repository

*Pairs · 20 min (whole team)*

> Set up your team repository from the template with `.github/workflows/ci.yml` and a ruleset for `main`: pull request required, one approval, status check `test` required, no force pushes. Open one deliberately failing pull request (for example with an unused import) and check that it cannot be merged.

*Source:* original exercise for this course (workspace exercise 8); background: [GitHub Docs: About protected branches](https://docs.github.com/en/repositories/configuring-branches-and-merges-in-your-repository/managing-protected-branches/about-protected-branches).

*Your answer:*

### 2.6 · *Stretch:* Price per guest (workspace)

*Code · 15 min*

> Add a method `price_per_guest()` to `ListingTable` that divides the short-stay price by `accommodates` and returns the median per room type. Add `accommodates` to the test data in `tests/conftest.py`, and write a test with a value you computed by hand. Should `accommodates` become a required column?

*Source:* original exercise for this course (workspace exercise 9).

*Your answer:*

## Debrief (15 min)

- **Commit graphs.** Compare the graphs drawn in 2.1 with `git log --oneline --graph --all`. Did anyone predict that Git would choose one version on its own?
- **Typical mistake: a marker left behind.** Look for a resolved file that still contains `<<<<<<<`, `=======` or `>>>>>>>`, and for a merge where one person's change silently disappeared because "Accept Current" was clicked without reading.
- **Notebook against module.** How many places had to change for the 30-night rule in each version (2.2)?
- **Reviews.** Read one review comment from 2.4 aloud: does it name a problem and propose a fix? Was the merge conflict resolved on the branch or on `main`?
- **Workspace numbers.** Why is the median number of reviews so much lower than the mean (2.3)?
- **Bridge to Session 3.** From now on, all team work on the final project is submitted as reviewed pull requests with passing CI. Session 3 moves the data itself into a relational database and queries it with SQL.

## Further reading

- The Alan Turing Institute. *Research Software Engineering with Python*, modules 4 (version control with Git), 5 (continuous integration), 6 (software projects) and 7 (linting). CC BY 3.0. https://alan-turing-institute.github.io/rse-course/ (extras 14–23 of this session).
- Chacon, S., & Straub, B. (2014). *Pro Git*, 2nd edition, chapters 2, 3 and 6. Apress; free online, CC BY-NC-SA 3.0. https://git-scm.com/book/en/v2
- Software Carpentry (2024). *Version Control with Git*. CC BY 4.0. https://swcarpentry.github.io/git-novice/
- CodeRefinery (2025). *Collaborative distributed version control*. CC BY 4.0. https://coderefinery.github.io/git-collaborative/
- MIT CSAIL (2026). *The Missing Semester of Your CS Education: Version control*. CC BY-NC-SA 4.0. https://missing.csail.mit.edu/2026/version-control/
- Google (2026). *Google Engineering Practices: Code review*. https://google.github.io/eng-practices/review/
- Python Packaging Authority (2026). *src layout vs flat layout*. https://packaging.python.org/en/latest/discussions/src-layout-vs-flat-layout/
- Astral (2026). *uv documentation: Working on projects*; *ruff documentation*. https://docs.astral.sh/uv/guides/projects/, https://docs.astral.sh/ruff/
- GitHub (2026). *GitHub Actions documentation*; *About protected branches*. https://docs.github.com/en/actions
- Wilson, G., Bryan, J., Cranston, K., Kitzes, J., Nederbragt, L., & Teal, T. K. (2017). Good enough practices in scientific computing. *PLOS Computational Biology*, 13(6), e1005510. https://doi.org/10.1371/journal.pcbi.1005510
- Interactive practice: [Learn Git Branching](https://learngitbranching.js.org/) and [GitHub Skills: Review pull requests](https://github.com/skills/review-pull-requests).

Provenance of all extras: [extras/README.md](extras/README.md).